In [27]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/hse-rec-sys-challenge-2026/user_features.csv
/kaggle/input/competitions/hse-rec-sys-challenge-2026/item_features.csv
/kaggle/input/competitions/hse-rec-sys-challenge-2026/submission_sample.csv
/kaggle/input/competitions/hse-rec-sys-challenge-2026/events.csv


In [2]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('hse-rec-sys-challenge-2026')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/hse-rec-sys-challenge-2026


In [3]:
import pandas as pd
import numpy as np

path = '/kaggle/input/competitions/hse-rec-sys-challenge-2026'

events = pd.read_csv(f'{path}/events.csv')
user_features = pd.read_csv(f'{path}/user_features.csv')
item_features = pd.read_csv(f'{path}/item_features.csv')
submission_sample = pd.read_csv(f'{path}/submission_sample.csv')

print("events:", events.shape)
print(events.head())
print("\nsubmission_sample:", submission_sample.shape)
print(submission_sample.head())
print("\nuser_features:", user_features.shape)
print(user_features.columns.tolist())
print("\nitem_features:", item_features.shape)
print(item_features.columns.tolist())
print("\nevents stats:")
print(events.describe())
print("\nUnique users:", events['user_id'].nunique())
print("Unique items:", events['item_id'].nunique())

events: (894149, 4)
   user_id  item_id  rating  timestamp
0        0     1505       4          0
1        0     3669       3          1
2        0      584       4          2
3        0     3390       3          3
4        0     2885       4          4

submission_sample: (6040, 2)
   user_id              item_id
0        0  0 1 2 3 4 5 6 7 8 9
1        1  0 1 2 3 4 5 6 7 8 9
2        2  0 1 2 3 4 5 6 7 8 9
3        3  0 1 2 3 4 5 6 7 8 9
4        4  0 1 2 3 4 5 6 7 8 9

user_features: (6040, 3)
['user_id', 'gender', 'age']

item_features: (3706, 19)
['item_id', 'genre_0', 'genre_1', 'genre_2', 'genre_3', 'genre_4', 'genre_5', 'genre_6', 'genre_7', 'genre_8', 'genre_9', 'genre_10', 'genre_11', 'genre_12', 'genre_13', 'genre_14', 'genre_15', 'genre_16', 'genre_17']

events stats:
             user_id        item_id         rating      timestamp
count  894149.000000  894149.000000  894149.000000  894149.000000
mean     3000.009516    1844.262022       3.580947     194.684738
std      17

 # Bert4Sec

In [4]:
!pip install -q replay-rec lightning pyarrow

import replay
import torch
import lightning


In [6]:
import pandas as pd
import torch
import lightning as L

from collections import defaultdict
from IPython.display import FileLink, display
from torch.utils.data import DataLoader

from replay.data import (
    FeatureHint,
    FeatureInfo,
    FeatureSchema,
    FeatureSource,
    FeatureType,
    Dataset,
)
from replay.data.nn import (
    SequenceTokenizer,
    SequentialDataset,
    TensorFeatureInfo,
    TensorFeatureSource,
    TensorSchema,
)
from replay.metrics import OfflineMetrics, Recall
from replay.metrics.torch_metrics_builder import metrics_to_df
from replay.models.nn.optimizer_utils import FatOptimizerFactory
from replay.models.nn.sequential import Bert4Rec
from replay.models.nn.sequential.bert4rec import (
    Bert4RecPredictionDataset,
    Bert4RecTrainingDataset,
)
from replay.models.nn.sequential.callbacks import PandasPredictionCallback
from replay.models.nn.sequential.postprocessors import RemoveSeenItems
from replay.splitters import LastNSplitter

L.seed_everything(50, workers=True)

interactions = (
    events[["user_id", "item_id", "timestamp"]]
    .sort_values(["user_id", "timestamp"])
    .copy()
)

interactions["timestamp"] = interactions["timestamp"].astype("int64")

Seed set to 50


In [9]:
splitter = LastNSplitter(
    N=1,
    divide_column="user_id",
    query_column="user_id",
    strategy="interactions",
    drop_cold_users=True,
    drop_cold_items=True,
)

raw_test_events, raw_test_gt = splitter.split(interactions)
raw_validation_events, raw_validation_gt = splitter.split(raw_test_events)
raw_train_events = raw_validation_events

def interaction_schema(with_timestamp=True):
    features = [
        FeatureInfo(
            column="user_id",
            feature_hint=FeatureHint.QUERY_ID,
            feature_type=FeatureType.CATEGORICAL,
        ),
        FeatureInfo(
            column="item_id",
            feature_hint=FeatureHint.ITEM_ID,
            feature_type=FeatureType.CATEGORICAL,
        ),
    ]

    if with_timestamp:
        features.append(
            FeatureInfo(
                column="timestamp",
                feature_hint=FeatureHint.TIMESTAMP,
                feature_type=FeatureType.NUMERICAL,
            )
        )

    return FeatureSchema(features)

train_dataset = Dataset(
    feature_schema=interaction_schema(with_timestamp=True),
    interactions=raw_train_events,
    check_consistency=True,
    categorical_encoded=False,
)

validation_dataset = Dataset(
    feature_schema=interaction_schema(with_timestamp=True),
    interactions=raw_validation_events,
    check_consistency=True,
    categorical_encoded=False,
)

validation_gt = Dataset(
    feature_schema=interaction_schema(with_timestamp=False),
    interactions=raw_validation_gt[["user_id", "item_id"]],
    check_consistency=True,
    categorical_encoded=False,
)

# Полная история нужна позже для финального submission.
full_dataset = Dataset(
    feature_schema=interaction_schema(with_timestamp=True),
    interactions=interactions,
    check_consistency=True,
    categorical_encoded=False,
)

print("Train:", raw_train_events.shape)
print("Validation target:", raw_validation_gt.shape)
print("Test target:", raw_test_gt.shape)

Train: (882069, 3)
Validation target: (6040, 3)
Test target: (6039, 3)


##  обычный берт

In [10]:

BASE_ITEM_FEATURE = "item_id_seq"
BASE_EMBEDDING_DIM = 128
BASE_MAX_SEQ_LEN = 100

tensor_schema_base = TensorSchema(
    TensorFeatureInfo(
        name=BASE_ITEM_FEATURE,
        is_seq=True,
        feature_type=FeatureType.CATEGORICAL,
        feature_sources=[
            TensorFeatureSource(
                FeatureSource.INTERACTIONS,
                train_dataset.feature_schema.item_id_column,
            )
        ],
        feature_hint=FeatureHint.ITEM_ID,
        embedding_dim=BASE_EMBEDDING_DIM,
    )
)

tokenizer_base = SequenceTokenizer(
    tensor_schema_base,
    allow_collect_to_master=True,
)

tokenizer_base.fit(full_dataset)

seq_train_base = tokenizer_base.transform(train_dataset)
seq_val_base = tokenizer_base.transform(validation_dataset)

seq_val_gt_base = tokenizer_base.transform(
    validation_gt,
    [tensor_schema_base.item_id_feature_name],
)

seq_val_base, seq_val_gt_base = SequentialDataset.keep_common_query_ids(
    seq_val_base,
    seq_val_gt_base,
)

seq_full_base = tokenizer_base.transform(full_dataset)

print("Base BERT prepared")
print("Train users:", raw_train_events["user_id"].nunique())
print("Items:", interactions["item_id"].nunique())

/tmp/ipykernel_48/3455253695.py:29: DeprecationWarning: `SequenceTokenizer` class is deprecated.
  tokenizer_base = SequenceTokenizer(


Base BERT prepared
Train users: 6040
Items: 3690


In [11]:
L.seed_everything(42, workers=True)

BASE_BATCH_SIZE = 512
BASE_NUM_WORKERS = 0
BASE_EPOCHS = 500

bert4rec_base = Bert4Rec(
    tensor_schema=tensor_schema_base,
    block_count=2,
    head_count=4,
    max_seq_len=BASE_MAX_SEQ_LEN,
    hidden_size=128,
    dropout_rate=0.2,
    optimizer_factory=FatOptimizerFactory(learning_rate=0.001),
)

train_loader_base = DataLoader(
    Bert4RecTrainingDataset(
        seq_train_base,
        max_sequence_length=BASE_MAX_SEQ_LEN,
    ),
    batch_size=BASE_BATCH_SIZE,
    shuffle=True,
    num_workers=BASE_NUM_WORKERS,
    pin_memory=True,
)

trainer_base = L.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=BASE_EPOCHS,
    logger=False,
    enable_checkpointing=False,
    log_every_n_steps=1,
)

trainer_base.fit(
    bert4rec_base,
    train_dataloaders=train_loader_base,
)

base_checkpoint = "/kaggle/working/bert4rec_base_128.ckpt"
trainer_base.save_checkpoint(base_checkpoint)

Seed set to 42
/tmp/ipykernel_48/4033216254.py:19: DeprecationWarning: `FatOptimizerFactory` class is deprecated. Use `replay.nn.lightning.optimizer.OptimizerFactory` instead.
  optimizer_factory=FatOptimizerFactory(learning_rate=0.001),
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name   ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _model │ Bert4RecModel    │  1.4 M │ train │     0 │
│ 1 │ _loss  │ CrossEntropyLoss │      0 │ train │     0 │
└───┴────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 1.4 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.4 M                                                                                                
Total estimated model params size (MB): 5.431                                                                      
Modules in train mode: 38                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
`Trainer.fit` stopped: `max_epochs=500` reached.


`weights_only` was not set, defaulting to `False`.


In [12]:
base_model = Bert4Rec.load_from_checkpoint(
    "/kaggle/working/bert4rec_base_128.ckpt",
    weights_only=False,
)

val_loader_base = DataLoader(
    Bert4RecPredictionDataset(
        seq_val_base,
        max_sequence_length=BASE_MAX_SEQ_LEN,
    ),
    batch_size=BASE_BATCH_SIZE,
    num_workers=BASE_NUM_WORKERS,
    pin_memory=True,
)

base_callback = PandasPredictionCallback(
    top_k=10
    query_column="user_id",
    item_column="item_id",
    rating_column="score",
    postprocessors=[RemoveSeenItems(seq_val_base)],
)

base_eval_trainer = L.Trainer(
    accelerator="gpu",
    devices=1,
    callbacks=[base_callback],
    logger=False,
    inference_mode=True,
)

base_eval_trainer.predict(
    base_model,
    dataloaders=val_loader_base,
    return_predictions=False,
)

base_val_predictions = tokenizer_base.query_and_item_id_encoder.inverse_transform(
    base_callback.get_result()
)

base_metrics = OfflineMetrics(
    [Recall([10])],
    query_column="user_id",
    rating_column="score",
)(
    base_val_predictions,
    raw_validation_gt,
)

metrics_to_df(base_metrics)

/tmp/ipykernel_48/3739176733.py:27: DeprecationWarning: `RemoveSeenItems` class is deprecated. Use `replay.nn.lightning.postprocessor.SeenItemsFilter` instead.
  postprocessors=[RemoveSeenItems(seq_val_base)],
/tmp/ipykernel_48/3739176733.py:22: DeprecationWarning: `PandasPredictionCallback` class is deprecated. Use `replay.nn.lightning.callback.PandasTopItemsCallback` instead.
  base_callback = PandasPredictionCallback(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


k,10
Recall,0.286258


## большой берт

In [57]:


ITEM_FEATURE_NAME = "item_id_seq"

tensor_schema = TensorSchema(
    TensorFeatureInfo(
        name=ITEM_FEATURE_NAME,
        is_seq=True,
        feature_type=FeatureType.CATEGORICAL,
        feature_sources=[
            TensorFeatureSource(
                FeatureSource.INTERACTIONS,
                train_dataset.feature_schema.item_id_column,
            )
        ],
        feature_hint=FeatureHint.ITEM_ID,
        embedding_dim=256,
    )
)

tokenizer = SequenceTokenizer(
    tensor_schema,
    allow_collect_to_master=True,
)

tokenizer.fit(full_dataset)

sequential_train_dataset = tokenizer.transform(train_dataset)

sequential_validation_dataset = tokenizer.transform(validation_dataset)

sequential_validation_gt = tokenizer.transform(
    validation_gt,
    [tensor_schema.item_id_feature_name],
)

(
    sequential_validation_dataset,
    sequential_validation_gt,
) = SequentialDataset.keep_common_query_ids(
    sequential_validation_dataset,
    sequential_validation_gt,
)

sequential_full_dataset = tokenizer.transform(full_dataset)

print("Train users:", raw_train_events["user_id"].nunique())
print("Validation users:", raw_validation_events["user_id"].nunique())
print("Items in tokenizer:", interactions["item_id"].nunique())

/tmp/ipykernel_49/1830740939.py:27: DeprecationWarning: `SequenceTokenizer` class is deprecated.
  tokenizer = SequenceTokenizer(


Train users: 6040
Validation users: 6040
Items in tokenizer: 3690


In [58]:
MAX_SEQ_LEN = 100
BATCH_SIZE = 512
NUM_WORKERS = 4
MAX_EPOCHS = 300

bert4rec = Bert4Rec(
    tensor_schema=tensor_schema,
    block_count=3,
    head_count=4,
    max_seq_len=MAX_SEQ_LEN,
    hidden_size=256,
    dropout_rate=0.2,
    optimizer_factory=FatOptimizerFactory(learning_rate=0.001),
)

train_dataloader = DataLoader(
    Bert4RecTrainingDataset(
        sequential_train_dataset,
        max_sequence_length=MAX_SEQ_LEN,
    ),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

trainer = L.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=MAX_EPOCHS,
    logger=False,
    enable_checkpointing=False,
    log_every_n_steps=1,
)

trainer.fit(
    bert4rec,
    train_dataloaders=train_dataloader,
)

checkpoint_path = "/kaggle/working/bert4rec_replay_large.ckpt"
trainer.save_checkpoint(checkpoint_path)

print(f"Saved checkpoint: {checkpoint_path}")

/tmp/ipykernel_49/2605737782.py:13: DeprecationWarning: `FatOptimizerFactory` class is deprecated. Use `replay.nn.lightning.optimizer.OptimizerFactory` instead.
  optimizer_factory=FatOptimizerFactory(learning_rate=0.001),
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name   ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _model │ Bert4RecModel    │  4.3 M │ train │     0 │
│ 1 │ _loss  │ CrossEntropyLoss │      0 │ train │     0 │
└───┴────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 4.3 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.3 M                                                                                                
Total estimated model params size (MB): 17.152                                                                     
Modules in train mode: 51                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

`Trainer.fit` stopped: `max_epochs=300` reached.


`weights_only` was not set, defaulting to `False`.


Saved checkpoint: /kaggle/working/bert4rec_replay_large.ckpt


In [59]:

best_model = Bert4Rec.load_from_checkpoint(
    "/kaggle/working/bert4rec_replay_large.ckpt",
    weights_only=False,
)

validation_prediction_dataloader = DataLoader(
    Bert4RecPredictionDataset(
        sequential_validation_dataset,
        max_sequence_length=MAX_SEQ_LEN,
    ),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

validation_prediction_callback = PandasPredictionCallback(
    top_k=10,
    query_column="user_id",
    item_column="item_id",
    rating_column="score",
    postprocessors=[
        RemoveSeenItems(sequential_validation_dataset)
    ],
)

validation_trainer = L.Trainer(
    accelerator="gpu",
    devices=1,
    callbacks=[validation_prediction_callback],
    logger=False,
    inference_mode=True,
)

validation_trainer.predict(
    best_model,
    dataloaders=validation_prediction_dataloader,
    return_predictions=False,
)

encoded_validation_predictions = validation_prediction_callback.get_result()

validation_predictions = tokenizer.query_and_item_id_encoder.inverse_transform(
    encoded_validation_predictions
)

offline_metrics = OfflineMetrics(
    [Recall([10])],
    query_column="user_id",
    rating_column="score",
)(
    validation_predictions,
    raw_validation_gt,
)

metrics_to_df(offline_metrics)

/tmp/ipykernel_49/1868229336.py:25: DeprecationWarning: `RemoveSeenItems` class is deprecated. Use `replay.nn.lightning.postprocessor.SeenItemsFilter` instead.
  RemoveSeenItems(sequential_validation_dataset)
/tmp/ipykernel_49/1868229336.py:19: DeprecationWarning: `PandasPredictionCallback` class is deprecated. Use `replay.nn.lightning.callback.PandasTopItemsCallback` instead.
  validation_prediction_callback = PandasPredictionCallback(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Output()

k,10
Recall,0.268543


In [36]:

final_prediction_dataloader = DataLoader(
    Bert4RecPredictionDataset(
        sequential_full_dataset,
        max_sequence_length=MAX_SEQ_LEN,
    ),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

final_prediction_callback = PandasPredictionCallback(
    top_k=10,
    query_column="user_id",
    item_column="item_id",
    rating_column="score",
    postprocessors=[
        # Не рекомендуем уже просмотренные фильмы.
        RemoveSeenItems(sequential_full_dataset)
    ],
)

final_trainer = L.Trainer(
    accelerator="gpu",
    devices=1,
    callbacks=[final_prediction_callback],
    logger=False,
    inference_mode=True,
)

final_trainer.predict(
    best_model,
    dataloaders=final_prediction_dataloader,
    return_predictions=False,
)

encoded_predictions = final_prediction_callback.get_result()

# Переводим внутренние ID RePlay обратно в исходные item_id контеста.
predictions = tokenizer.query_and_item_id_encoder.inverse_transform(
    encoded_predictions
)

predictions = predictions.sort_values(
    ["user_id", "score"],
    ascending=[True, False],
)

submission = (
    predictions
    .groupby("user_id")["item_id"]
    .apply(lambda items: " ".join(map(str, items.tolist()[:10])))
    .reset_index()
)

# Сохраняем тот же порядок пользователей, что в submission_sample.
submission = (
    submission_sample[["user_id"]]
    .merge(submission, on="user_id", how="left")
)

output_path = "/kaggle/working/submission_bert4rec_replay3.csv"
submission.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
display(submission.head())
display(FileLink(output_path))

/tmp/ipykernel_49/4254744305.py:24: DeprecationWarning: `RemoveSeenItems` class is deprecated. Use `replay.nn.lightning.postprocessor.SeenItemsFilter` instead.
  RemoveSeenItems(sequential_full_dataset)
/tmp/ipykernel_49/4254744305.py:17: DeprecationWarning: `PandasPredictionCallback` class is deprecated. Use `replay.nn.lightning.callback.PandasTopItemsCallback` instead.
  final_prediction_callback = PandasPredictionCallback(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/

Output()

Saved: /kaggle/working/submission_bert4rec_replay3.csv


,user_id,item_id
0,0,1461 3002 1171 2145 128 1044 708 1422 2003 2275
1,1,232 1686 2336 3656 2387 452 1868 3311 3101 463
2,2,2354 382 2774 3687 2311 2424 1947 1687 724 234
3,3,79 232 3035 989 467 1868 3656 1754 1545 487
4,4,1545 2814 1337 802 2185 2653 1086 3035 702 488


/kaggle/working/submission_bert4rec_replay3.csv

## Данные с рейтингом

In [50]:
interactions_rating = (
    events[["user_id", "item_id", "rating", "timestamp"]]
    .sort_values(["user_id", "timestamp"])
    .copy()
)

def schema_with_rating(with_timestamp=True, with_rating=True):
    features = [
        FeatureInfo(
            column="user_id",
            feature_hint=FeatureHint.QUERY_ID,
            feature_type=FeatureType.CATEGORICAL,
        ),
        FeatureInfo(
            column="item_id",
            feature_hint=FeatureHint.ITEM_ID,
            feature_type=FeatureType.CATEGORICAL,
        ),
    ]

    if with_rating:
        features.append(
            FeatureInfo(
                column="rating",
                feature_type=FeatureType.CATEGORICAL,
            )
        )

    if with_timestamp:
        features.append(
            FeatureInfo(
                column="timestamp",
                feature_hint=FeatureHint.TIMESTAMP,
                feature_type=FeatureType.NUMERICAL,
            )
        )

    return FeatureSchema(features)

splitter = LastNSplitter(
    N=1,
    divide_column="user_id",
    query_column="user_id",
    strategy="interactions",
    drop_cold_users=True,
    drop_cold_items=True,
)

test_events_rating, test_gt_rating = splitter.split(interactions_rating)
val_events_rating, val_gt_rating = splitter.split(test_events_rating)
train_events_rating = val_events_rating

train_dataset_rating = Dataset(
    feature_schema=schema_with_rating(),
    interactions=train_events_rating,
    check_consistency=True,
    categorical_encoded=False,
)

val_dataset_rating = Dataset(
    feature_schema=schema_with_rating(),
    interactions=val_events_rating,
    check_consistency=True,
    categorical_encoded=False,
)

val_gt_dataset_rating = Dataset(
    feature_schema=schema_with_rating(
        with_timestamp=False,
        with_rating=False,
    ),
    interactions=val_gt_rating[["user_id", "item_id"]],
    check_consistency=True,
    categorical_encoded=False,
)

full_dataset_rating = Dataset(
    feature_schema=schema_with_rating(),
    interactions=interactions_rating,
    check_consistency=True,
    categorical_encoded=False,
)

print("Train:", train_events_rating.shape)
print("Validation:", val_gt_rating.shape)

Train: (882069, 4)
Validation: (6040, 4)


In [51]:
ITEM_FEATURE_NAME = "item_id_seq"
RATING_FEATURE_NAME = "rating_seq"

tensor_schema_rating = TensorSchema([
    TensorFeatureInfo(
        name=ITEM_FEATURE_NAME,
        is_seq=True,
        feature_type=FeatureType.CATEGORICAL,
        feature_sources=[
            TensorFeatureSource(
                FeatureSource.INTERACTIONS,
                train_dataset_rating.feature_schema.item_id_column,
            )
        ],
        feature_hint=FeatureHint.ITEM_ID,
        embedding_dim=128,
    ),
    TensorFeatureInfo(
        name=RATING_FEATURE_NAME,
        is_seq=True,
        feature_type=FeatureType.CATEGORICAL,
        feature_sources=[
            TensorFeatureSource(
                FeatureSource.INTERACTIONS,
                "rating",
            )
        ],
        embedding_dim=128,
    ),
])

tokenizer_rating = SequenceTokenizer(
    tensor_schema_rating,
    allow_collect_to_master=True,
)

# Fit по полной истории нужен, чтобы tokenizer знал все ID фильмов.
tokenizer_rating.fit(full_dataset_rating)

seq_train_rating = tokenizer_rating.transform(train_dataset_rating)
seq_val_rating = tokenizer_rating.transform(val_dataset_rating)

seq_val_gt_rating = tokenizer_rating.transform(
    val_gt_dataset_rating,
    [tensor_schema_rating.item_id_feature_name],
)

seq_val_rating, seq_val_gt_rating = SequentialDataset.keep_common_query_ids(
    seq_val_rating,
    seq_val_gt_rating,
)

seq_full_rating = tokenizer_rating.transform(full_dataset_rating)

print("Готово. Пользователей в train:", train_events_rating["user_id"].nunique())
print("Фильмов:", interactions_rating["item_id"].nunique())

/tmp/ipykernel_49/345972615.py:32: DeprecationWarning: `SequenceTokenizer` class is deprecated.
  tokenizer_rating = SequenceTokenizer(


Готово. Пользователей в train: 6040
Фильмов: 3690


In [52]:
MAX_SEQ_LEN = 100
BATCH_SIZE = 512
NUM_WORKERS = 0       # безопаснее: не зависает в Kaggle
MAX_EPOCHS = 300      # сначала проверяем 150, не 500

bert4rec_rating = Bert4Rec(
    tensor_schema=tensor_schema_rating,
    block_count=2,
    head_count=4,
    max_seq_len=MAX_SEQ_LEN,
    hidden_size=128,
    dropout_rate=0.2,
    optimizer_factory=FatOptimizerFactory(learning_rate=0.001),
)

train_loader_rating = DataLoader(
    Bert4RecTrainingDataset(
        seq_train_rating,
        max_sequence_length=MAX_SEQ_LEN,
    ),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

trainer_rating = L.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=MAX_EPOCHS,
    logger=False,
    enable_checkpointing=False,
    log_every_n_steps=1,
)

trainer_rating.fit(
    bert4rec_rating,
    train_dataloaders=train_loader_rating,
)

rating_checkpoint = "/kaggle/working/bert4rec_with_rating.ckpt"
trainer_rating.save_checkpoint(rating_checkpoint)

print(f"Saved: {rating_checkpoint}")

/tmp/ipykernel_49/2755633278.py:19: DeprecationWarning: `FatOptimizerFactory` class is deprecated. Use `replay.nn.lightning.optimizer.OptimizerFactory` instead.
  optimizer_factory=FatOptimizerFactory(learning_rate=0.001),
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name   ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _model │ Bert4RecModel    │  1.4 M │ train │     0 │
│ 1 │ _loss  │ CrossEntropyLoss │      0 │ train │     0 │
└───┴────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 1.4 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.4 M                                                                                                
Total estimated model params size (MB): 5.434                                                                      
Modules in train mode: 39                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
`Trainer.fit` stopped: `max_epochs=300` reached.


`weights_only` was not set, defaulting to `False`.


Saved: /kaggle/working/bert4rec_with_rating.ckpt


In [53]:
best_rating_model = Bert4Rec.load_from_checkpoint(
    "/kaggle/working/bert4rec_with_rating.ckpt",
    weights_only=False,
)

val_loader_rating = DataLoader(
    Bert4RecPredictionDataset(
        seq_val_rating,
        max_sequence_length=MAX_SEQ_LEN,
    ),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_callback_rating = PandasPredictionCallback(
    top_k=10,
    query_column="user_id",
    item_column="item_id",
    rating_column="score",
    postprocessors=[
        RemoveSeenItems(seq_val_rating)
    ],
)

val_trainer_rating = L.Trainer(
    accelerator="gpu",
    devices=1,
    callbacks=[val_callback_rating],
    logger=False,
    inference_mode=True,
)

val_trainer_rating.predict(
    best_rating_model,
    dataloaders=val_loader_rating,
    return_predictions=False,
)

encoded_val_predictions_rating = val_callback_rating.get_result()

val_predictions_rating = (
    tokenizer_rating.query_and_item_id_encoder.inverse_transform(
        encoded_val_predictions_rating
    )
)

rating_metrics = OfflineMetrics(
    [Recall([10])],
    query_column="user_id",
    rating_column="score",
)(
    val_predictions_rating,
    val_gt_rating,
)

metrics_to_df(rating_metrics)

/tmp/ipykernel_49/1788758504.py:28: DeprecationWarning: `RemoveSeenItems` class is deprecated. Use `replay.nn.lightning.postprocessor.SeenItemsFilter` instead.
  RemoveSeenItems(seq_val_rating)
/tmp/ipykernel_49/1788758504.py:22: DeprecationWarning: `PandasPredictionCallback` class is deprecated. Use `replay.nn.lightning.callback.PandasTopItemsCallback` instead.
  val_callback_rating = PandasPredictionCallback(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


k,10
Recall,0.288245


In [43]:
final_loader_rating = DataLoader(
    Bert4RecPredictionDataset(
        seq_full_rating,
        max_sequence_length=MAX_SEQ_LEN,
    ),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

final_callback_rating = PandasPredictionCallback(
    top_k=10,
    query_column="user_id",
    item_column="item_id",
    rating_column="score",
    postprocessors=[
        RemoveSeenItems(seq_full_rating)
    ],
)

final_trainer_rating = L.Trainer(
    accelerator="gpu",
    devices=1,
    callbacks=[final_callback_rating],
    logger=False,
    inference_mode=True,
)

final_trainer_rating.predict(
    best_rating_model,
    dataloaders=final_loader_rating,
    return_predictions=False,
)

encoded_final_predictions_rating = final_callback_rating.get_result()

final_predictions_rating = (
    tokenizer_rating.query_and_item_id_encoder.inverse_transform(
        encoded_final_predictions_rating
    )
)

final_predictions_rating = final_predictions_rating.sort_values(
    ["user_id", "score"],
    ascending=[True, False],
)

submission_rating = (
    final_predictions_rating
    .groupby("user_id")["item_id"]
    .apply(lambda items: " ".join(map(str, items.tolist()[:10])))
    .reset_index()
)

submission_rating = (
    submission_sample[["user_id"]]
    .merge(submission_rating, on="user_id", how="left")
)

output_path = "/kaggle/working/submission_bert4rec_rating.csv"
submission_rating.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
display(submission_rating.head())

/tmp/ipykernel_49/3854563834.py:19: DeprecationWarning: `RemoveSeenItems` class is deprecated. Use `replay.nn.lightning.postprocessor.SeenItemsFilter` instead.
  RemoveSeenItems(seq_full_rating)
/tmp/ipykernel_49/3854563834.py:13: DeprecationWarning: `PandasPredictionCallback` class is deprecated. Use `replay.nn.lightning.callback.PandasTopItemsCallback` instead.
  final_callback_rating = PandasPredictionCallback(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Saved: /kaggle/working/submission_bert4rec_rating.csv


,user_id,item_id
0,0,3002 163 1461 2275 1044 3087 213 3411 1100 2145
1,1,3656 452 1686 232 3101 3311 1868 1246 528 1573
2,2,234 382 2428 2354 2774 221 2311 1560 1687 724
3,3,3656 1545 232 1691 3600 802 2814 3406 3035 79
4,4,1545 2185 802 1086 1337 2814 2653 2205 702 1687


/kaggle/working/submission_bert4rec_rating.csv

## ансамбль бертов

In [54]:
# Загружаем обе модели.
base_ensemble_model = Bert4Rec.load_from_checkpoint(
    "/kaggle/working/bert4rec_base_128.ckpt",
    weights_only=False,
)

rating_ensemble_model = Bert4Rec.load_from_checkpoint(
    "/kaggle/working/bert4rec_with_rating.ckpt",
    weights_only=False,
)

def get_top100_predictions(model, sequential_dataset, tokenizer):
    loader = DataLoader(
        Bert4RecPredictionDataset(
            sequential_dataset,
            max_sequence_length=MAX_SEQ_LEN,
        ),
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    callback = PandasPredictionCallback(
        top_k=100,
        query_column="user_id",
        item_column="item_id",
        rating_column="score",
        postprocessors=[
            RemoveSeenItems(sequential_dataset)
        ],
    )

    predict_trainer = L.Trainer(
        accelerator="gpu",
        devices=1,
        callbacks=[callback],
        logger=False,
        inference_mode=True,
    )

    predict_trainer.predict(
        model,
        dataloaders=loader,
        return_predictions=False,
    )

    encoded = callback.get_result()

    return tokenizer.query_and_item_id_encoder.inverse_transform(encoded)

# Предсказания на одной и той же validation-выборке.
base_val_top100 = get_top100_predictions(
    base_ensemble_model,
    seq_val_base,
    tokenizer_base,
)

rating_val_top100 = get_top100_predictions(
    rating_ensemble_model,
    seq_val_rating,
    tokenizer_rating,
)

base_lists = (
    base_val_top100
    .sort_values(["user_id", "score"], ascending=[True, False])
    .groupby("user_id")["item_id"]
    .apply(list)
    .to_dict()
)

rating_lists = (
    rating_val_top100
    .sort_values(["user_id", "score"], ascending=[True, False])
    .groupby("user_id")["item_id"]
    .apply(list)
    .to_dict()
)

val_targets = (
    raw_validation_gt
    .groupby("user_id")["item_id"]
    .first()
    .to_dict()
)

def rrf_merge(base_items, rating_items, rating_weight, k=10, rrf_k=60):
    scores = defaultdict(float)

    for rank, item_id in enumerate(base_items, start=1):
        scores[item_id] += (1 - rating_weight) / (rrf_k + rank)

    for rank, item_id in enumerate(rating_items, start=1):
        scores[item_id] += rating_weight / (rrf_k + rank)

    return [
        item_id
        for item_id, _ in sorted(
            scores.items(),
            key=lambda pair: pair[1],
            reverse=True,
        )[:k]
    ]

def recall_at_10(predictions, targets):
    common_users = [
        user_id
        for user_id in targets
        if user_id in predictions
    ]

    return np.mean([
        targets[user_id] in predictions[user_id][:10]
        for user_id in common_users
    ])

print("Base:", recall_at_10(base_lists, val_targets))
print("Rating:", recall_at_10(rating_lists, val_targets))

best_weight = 0.0
best_score = -1

for rating_weight in [0.0, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]:
    ensemble_lists = {
        user_id: rrf_merge(
            base_lists.get(user_id, []),
            rating_lists.get(user_id, []),
            rating_weight=rating_weight,
        )
        for user_id in val_targets
    }

    score = recall_at_10(ensemble_lists, val_targets)

    print(f"Rating weight={rating_weight:.2f} | Recall@10={score:.6f}")

    if score > best_score:
        best_score = score
        best_weight = rating_weight

print(f"\nBEST: Recall@10={best_score:.6f}, rating weight={best_weight:.2f}")

/tmp/ipykernel_49/1482905907.py:31: DeprecationWarning: `RemoveSeenItems` class is deprecated. Use `replay.nn.lightning.postprocessor.SeenItemsFilter` instead.
  RemoveSeenItems(sequential_dataset)
/tmp/ipykernel_49/1482905907.py:25: DeprecationWarning: `PandasPredictionCallback` class is deprecated. Use `replay.nn.lightning.callback.PandasTopItemsCallback` instead.
  callback = PandasPredictionCallback(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


/tmp/ipykernel_49/1482905907.py:31: DeprecationWarning: `RemoveSeenItems` class is deprecated. Use `replay.nn.lightning.postprocessor.SeenItemsFilter` instead.
  RemoveSeenItems(sequential_dataset)
/tmp/ipykernel_49/1482905907.py:25: DeprecationWarning: `PandasPredictionCallback` class is deprecated. Use `replay.nn.lightning.callback.PandasTopItemsCallback` instead.
  callback = PandasPredictionCallback(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Base: 0.29089403973509936
Rating: 0.28824503311258276
Rating weight=0.00 | Recall@10=0.290894
Rating weight=0.05 | Recall@10=0.291391
Rating weight=0.10 | Recall@10=0.291556
Rating weight=0.15 | Recall@10=0.292384
Rating weight=0.20 | Recall@10=0.293709
Rating weight=0.30 | Recall@10=0.294536
Rating weight=0.40 | Recall@10=0.296689
Rating weight=0.50 | Recall@10=0.296192

BEST: Recall@10=0.296689, rating weight=0.40


In [55]:
# Предсказания обеих моделей на полной истории пользователей.
base_final_top100 = get_top100_predictions(
    base_ensemble_model,
    seq_full_base,
    tokenizer_base,
)

rating_final_top100 = get_top100_predictions(
    rating_ensemble_model,
    seq_full_rating,
    tokenizer_rating,
)

base_final_lists = (
    base_final_top100
    .sort_values(["user_id", "score"], ascending=[True, False])
    .groupby("user_id")["item_id"]
    .apply(list)
    .to_dict()
)

rating_final_lists = (
    rating_final_top100
    .sort_values(["user_id", "score"], ascending=[True, False])
    .groupby("user_id")["item_id"]
    .apply(list)
    .to_dict()
)

RATING_WEIGHT = 0.40

submission_rows = []

for user_id in submission_sample["user_id"]:
    recommendations = rrf_merge(
        base_final_lists.get(user_id, []),
        rating_final_lists.get(user_id, []),
        rating_weight=RATING_WEIGHT,
        k=10,
    )


    submission_rows.append({
        "user_id": user_id,
        "item_id": " ".join(map(str, recommendations)),
    })

submission_ensemble = pd.DataFrame(submission_rows)


output_path = "/kaggle/working/submission_bert4rec_ensemble2.csv"
submission_ensemble.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
display(submission_ensemble.head())

/tmp/ipykernel_49/1482905907.py:31: DeprecationWarning: `RemoveSeenItems` class is deprecated. Use `replay.nn.lightning.postprocessor.SeenItemsFilter` instead.
  RemoveSeenItems(sequential_dataset)
/tmp/ipykernel_49/1482905907.py:25: DeprecationWarning: `PandasPredictionCallback` class is deprecated. Use `replay.nn.lightning.callback.PandasTopItemsCallback` instead.
  callback = PandasPredictionCallback(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


/tmp/ipykernel_49/1482905907.py:31: DeprecationWarning: `RemoveSeenItems` class is deprecated. Use `replay.nn.lightning.postprocessor.SeenItemsFilter` instead.
  RemoveSeenItems(sequential_dataset)
/tmp/ipykernel_49/1482905907.py:25: DeprecationWarning: `PandasPredictionCallback` class is deprecated. Use `replay.nn.lightning.callback.PandasTopItemsCallback` instead.
  callback = PandasPredictionCallback(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Saved: /kaggle/working/submission_bert4rec_ensemble2.csv


,user_id,item_id
0,0,3002 1461 2275 1171 1044 163 2593 128 3667 213
1,1,1686 232 2336 3656 1868 452 3311 1246 2357 1573
2,2,2354 382 1687 2774 2311 234 724 2428 887 221
3,3,232 79 3035 2363 802 3656 3187 989 1801 2345
4,4,802 1545 1337 2185 2653 2338 2814 1086 2205 570


## ансамбль 3 берта

In [60]:
# Offline RRF: base BERT + rating BERT + large BERT

large_ensemble_model = Bert4Rec.load_from_checkpoint(
    "/kaggle/working/bert4rec_replay_large.ckpt",
    weights_only=False,
)

large_val_top100 = get_top100_predictions(
    large_ensemble_model,
    sequential_validation_dataset,
    tokenizer,
)

large_lists = (
    large_val_top100
    .sort_values(["user_id", "score"], ascending=[True, False])
    .groupby("user_id")["item_id"]
    .apply(list)
    .to_dict()
)

def rrf_merge_three(
    base_items,
    rating_items,
    large_items,
    base_weight,
    rating_weight,
    large_weight,
    k=10,
    rrf_k=60,
):
    scores = defaultdict(float)

    for rank, item_id in enumerate(base_items, start=1):
        scores[item_id] += base_weight / (rrf_k + rank)

    for rank, item_id in enumerate(rating_items, start=1):
        scores[item_id] += rating_weight / (rrf_k + rank)

    for rank, item_id in enumerate(large_items, start=1):
        scores[item_id] += large_weight / (rrf_k + rank)

    return [
        item_id
        for item_id, _ in sorted(
            scores.items(),
            key=lambda pair: pair[1],
            reverse=True,
        )[:k]
    ]

best_config = None
best_score = -1

# Проверяем небольшой вклад large-модели:
# она слабее сама по себе, поэтому не даём ей большой вес.
for rating_weight in [0.25, 0.30, 0.35, 0.40, 0.45]:
    for large_weight in [0.00, 0.05, 0.10, 0.15, 0.20]:
        base_weight = 1.0 - rating_weight - large_weight

        if base_weight <= 0:
            continue

        three_model_predictions = {
            user_id: rrf_merge_three(
                base_lists.get(user_id, []),
                rating_lists.get(user_id, []),
                large_lists.get(user_id, []),
                base_weight=base_weight,
                rating_weight=rating_weight,
                large_weight=large_weight,
            )
            for user_id in val_targets
        }

        score = recall_at_10(
            three_model_predictions,
            val_targets,
        )

        print(
            f"base={base_weight:.2f}, "
            f"rating={rating_weight:.2f}, "
            f"large={large_weight:.2f} "
            f"| Recall@10={score:.6f}"
        )

        if score > best_score:
            best_score = score
            best_config = (
                base_weight,
                rating_weight,
                large_weight,
            )

print(
    "\nBEST THREE-MODEL ENSEMBLE:",
    f"Recall@10={best_score:.6f},",
    f"base={best_config[0]:.2f},",
    f"rating={best_config[1]:.2f},",
    f"large={best_config[2]:.2f}",
)

/tmp/ipykernel_49/1482905907.py:31: DeprecationWarning: `RemoveSeenItems` class is deprecated. Use `replay.nn.lightning.postprocessor.SeenItemsFilter` instead.
  RemoveSeenItems(sequential_dataset)
/tmp/ipykernel_49/1482905907.py:25: DeprecationWarning: `PandasPredictionCallback` class is deprecated. Use `replay.nn.lightning.callback.PandasTopItemsCallback` instead.
  callback = PandasPredictionCallback(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.13/dist-pa

Output()

base=0.75, rating=0.25, large=0.00 | Recall@10=0.294205
base=0.70, rating=0.25, large=0.05 | Recall@10=0.296192
base=0.65, rating=0.25, large=0.10 | Recall@10=0.299172
base=0.60, rating=0.25, large=0.15 | Recall@10=0.299834
base=0.55, rating=0.25, large=0.20 | Recall@10=0.298344
base=0.70, rating=0.30, large=0.00 | Recall@10=0.294536
base=0.65, rating=0.30, large=0.05 | Recall@10=0.296026
base=0.60, rating=0.30, large=0.10 | Recall@10=0.299834
base=0.55, rating=0.30, large=0.15 | Recall@10=0.297848
base=0.50, rating=0.30, large=0.20 | Recall@10=0.297848
base=0.65, rating=0.35, large=0.00 | Recall@10=0.295530
base=0.60, rating=0.35, large=0.05 | Recall@10=0.296026
base=0.55, rating=0.35, large=0.10 | Recall@10=0.297848
base=0.50, rating=0.35, large=0.15 | Recall@10=0.298013
base=0.45, rating=0.35, large=0.20 | Recall@10=0.298841
base=0.60, rating=0.40, large=0.00 | Recall@10=0.296689
base=0.55, rating=0.40, large=0.05 | Recall@10=0.296689
base=0.50, rating=0.40, large=0.10 | Recall@10=0

In [61]:
# Top-100 кандидатов всех трёх моделей на полной истории

base_final_top100 = get_top100_predictions(
    base_ensemble_model,
    seq_full_base,
    tokenizer_base,
)
rating_final_top100 = get_top100_predictions(
    rating_ensemble_model,
    seq_full_rating,
    tokenizer_rating,
)

large_final_top100 = get_top100_predictions(
    large_ensemble_model,
    sequential_full_dataset,
    tokenizer,
)

base_final_lists = (
    base_final_top100
    .sort_values(["user_id", "score"], ascending=[True, False])
    .groupby("user_id")["item_id"]
    .apply(list)
    .to_dict()
)

rating_final_lists = (
    rating_final_top100
    .sort_values(["user_id", "score"], ascending=[True, False])
    .groupby("user_id")["item_id"]
    .apply(list)
    .to_dict()
)

large_final_lists = (
    large_final_top100
    .sort_values(["user_id", "score"], ascending=[True, False])
    .groupby("user_id")["item_id"]
    .apply(list)
    .to_dict()
)

BASE_WEIGHT = 0.60
RATING_WEIGHT = 0.25
LARGE_WEIGHT = 0.15

submission_rows = []

for user_id in submission_sample["user_id"]:
    recommendations = rrf_merge_three(
        base_final_lists.get(user_id, []),
        rating_final_lists.get(user_id, []),
        large_final_lists.get(user_id, []),
        base_weight=BASE_WEIGHT,
        rating_weight=RATING_WEIGHT,
        large_weight=LARGE_WEIGHT,
        k=10,
    )

    assert len(recommendations) == 10
    assert len(set(recommendations)) == 10

    submission_rows.append({
        "user_id": user_id,
        "item_id": " ".join(map(str, recommendations)),
    })

submission_three_models = pd.DataFrame(submission_rows)

assert submission_three_models.shape == submission_sample.shape
assert submission_three_models["item_id"].str.split().str.len().eq(10).all()

output_path = "/kaggle/working/submission_bert4rec_three_models.csv"
submission_three_models.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
display(submission_three_models.head())

/tmp/ipykernel_49/1482905907.py:31: DeprecationWarning: `RemoveSeenItems` class is deprecated. Use `replay.nn.lightning.postprocessor.SeenItemsFilter` instead.
  RemoveSeenItems(sequential_dataset)
/tmp/ipykernel_49/1482905907.py:25: DeprecationWarning: `PandasPredictionCallback` class is deprecated. Use `replay.nn.lightning.callback.PandasTopItemsCallback` instead.
  callback = PandasPredictionCallback(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.13/dist-pa

Output()

/tmp/ipykernel_49/1482905907.py:31: DeprecationWarning: `RemoveSeenItems` class is deprecated. Use `replay.nn.lightning.postprocessor.SeenItemsFilter` instead.
  RemoveSeenItems(sequential_dataset)
/tmp/ipykernel_49/1482905907.py:25: DeprecationWarning: `PandasPredictionCallback` class is deprecated. Use `replay.nn.lightning.callback.PandasTopItemsCallback` instead.
  callback = PandasPredictionCallback(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.13/dist-pa

Output()

/tmp/ipykernel_49/1482905907.py:31: DeprecationWarning: `RemoveSeenItems` class is deprecated. Use `replay.nn.lightning.postprocessor.SeenItemsFilter` instead.
  RemoveSeenItems(sequential_dataset)
/tmp/ipykernel_49/1482905907.py:25: DeprecationWarning: `PandasPredictionCallback` class is deprecated. Use `replay.nn.lightning.callback.PandasTopItemsCallback` instead.
  callback = PandasPredictionCallback(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.13/dist-pa

Output()

Saved: /kaggle/working/submission_bert4rec_three_models.csv


,user_id,item_id
0,0,3002 2275 1461 1171 128 2593 1044 980 700 163
1,1,232 1686 3656 2336 3311 1868 452 1573 1849 3101
2,2,2354 382 1687 724 2774 234 2311 2428 887 3687
3,3,232 79 3035 2363 802 3656 3187 989 2345 1868
4,4,1545 1337 2185 802 2653 1086 2814 2338 570 488
